# Data wrangling
The goal of the functions presented in this document is to manipulate data obtained in previous steps

In [5]:
import pandas as pd

In [7]:
organismos = pd.read_csv("organismos.txt", sep=",")
k01220 = pd.read_csv("K01220.txt", sep=",")

Function `merge_dfs()` aims to combine data from both dataframes $\textbf{K01220}$ and $\textbf{organismos}$. For each element of the column $\textit{'Organisms'}$ in $\textbf{K01220}$ it searches into $\textit{'code'}$ column elements from $\textbf{organismos}$ to look for any coincidences. If both elements match, then the row of the element on $\textbf{organismos}$ should be saved. At the end of the entire function you should have those saved rows appended next to $\textbf{K01220}$

This was my first attempt to create the `merge_dfs()` function, it turns out that it works, but works on a time complexity $\textit{O(n · m)}$. Which results convenient when working with small data. If this was about just working with Python vanilla would be correct.

In [249]:
# def merge_dfs(k01220, organismos):
#     lista = []
#     for i in range(len(k01220)):
#         for j in range(len(organismos)):
#             if k01220["Organims"].loc[i] == organismos["code"].loc[j]:
#                 lista.append(organismos.loc[j])
#                 break 
#     return pd.DataFrame(lista)
# lista_orgs = merge_dfs(k01220, organismos)
# 
# lista_orgs = lista_orgs.drop(["entry", "code"], axis=1)  ##remove duplicate columns 
# k01220_phylo = pd.concat([k01220, lista_orgs], axis=1)   ## column-bind k01220 and resulting list  
# k01220_phylo.to_csv('k01220_phylo.txt', sep=",", index=False)

But since I'am using $\text{Pandas}$ python library, there is a method called $\text{pd.merge(df, df2, params...)}$ that does exactly the same than my previous function but with a radical change on time complexity, being this time $\textit{O(n + m)}$. 

The new `merge_dfs()` function implements this method an expects two dataframes and the name of two columns of those dataframes. This function takes the shared elements and their respective rows ant applyes some sort of $\text{"rbind"}$ from R. And outputs a dataframe with merged dfs.

In [18]:
def merge_dfs(k01220, organismos, left_col, right_col):
    return pd.merge(k01220, organismos, left_on=left_col, right_on=right_col, how='inner')

lista_orgs = merge_dfs(k01220, organismos, "Organims", "code")      # merge both dfs
lista_orgs = lista_orgs.drop(["entry", "code"], axis=1)             # delete a couple columns with useless info

In [ ]:
k01220_phylo.to_csv('k01220_phylo.txt', sep=",", index=False)       # create txt

The next function `clean_df()` the resulting dataframe from `merge_dfs()`, using this same dataframe we need to generate a `list` which will contain the column(s) name(s) that will be converted to a fasta header and also the column name of the desired sequences.   
The function `to_fasta()` uses the resulting dataframe from `clean_df()` and also requires a string that will serve as separator from the header and the sequence, typically fasta format uses a `'\n'` to achive this.  
The idea of `clean_df()` is to prepare data then give format with `to_fasta()` and later save as fasta.   

In [20]:
headers = ['Organims', 'id_Entry']
seqs = 'amino_Sequence'

def clean_df(lista_orgs, headers, seqs):
    headers = lista_orgs[headers].agg(' '.join, axis=1)                                       #joins content of desired columns and
    return pd.concat([headers, lista_orgs[seqs]], axis=1).rename(columns={0:'headers'})       #joins columns and rename a column

def to_fasta(headers_seqs, separator):
    if type(separator)==str:
        headers_seqs['headers'] = '>' + headers_seqs['headers']                               #append '>' before each element in column 'headers'
        headers_seqs['amino_Sequence'] = headers_seqs['amino_Sequence'] + '\n'                #adds a line break at the end of the sequence
        headers_export = headers_seqs.agg(separator.join, axis=1)                             #applies "".join a las filas   
        return separator.join(headers_export)                                                 #convierte la Series final a string con separator entre registrps

In [22]:
headers_seqs = clean_df(lista_orgs, headers, seqs)
final_fasta = to_fasta(headers_seqs, '\n')

In [27]:
with open('k01220.fas', 'w') as f:   #write fasta, might throw an error Errno 22, restart the entire kernel or close everything and restart
    f.write(final_fasta)